In [21]:
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd 
from pathlib import Path
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from google.colab import runtime
runtime.unassign()
from google.colab import drive
drive.mount('/content/drive')

MessageError: Failed to issue request GET https://colab.research.google.com/tun/m/credentials-propagation/gpu-a100-hm-kkb-euw4a1-81ouag98hmg1?authtype=dfs_ephemeral&version=2&dryrun=true&propagate=true&record=false&authuser=0: Not Found
Response body: 
<!DOCTYPE html>
<html lang=en>
  <meta charset=utf-8>
  <meta name=viewport content="initial-scale=1, minimum-scale=1, width=device-width">
  <title>Error 404 (Not Found)!!1</title>
  <style>
    *{margin:0;padding:0}html,code{font:15px/22px arial,sans-serif}html{background:#fff;color:#222;padding:15px}body{margin:7% auto 0;max-width:390px;min-height:180px;padding:30px 0 15px}* > body{background:url(//www.google.com/images/errors/robot.png) 100% 5px no-repeat;padding-right:205px}p{margin:11px 0 22px;overflow:hidden}ins{color:#777;text-decoration:none}a img{border:0}@media screen and (max-width:772px){body{background:none;margin-top:0;max-width:none;padding-right:0}}#logo{background:url(//www.google.com/images/logos/errorpage/error_logo-1

In [14]:

root_candidates = [
    Path.cwd() / "data" / "mallorn-astronomical-classification-challenge",
    Path("/content/drive/MyDrive/data/mallorn-astronomical-classification-challenge/"),
    Path("c:/Users/mlady/Documents/202620/stat6685/finalproject/data/mallorn-astronomical-classification-challenge"),
]
root = next((path for path in root_candidates if path.exists()), root_candidates[0])
# 1) Global training metadata with labels
train_log = pd.read_csv(root / "train_log.csv")
test_log = pd.read_csv(root / "test_log.csv")

# 2) Load one split's lightcurves and attach labels/features from global log
def load_split(split_name: str, is_test=False):
    if is_test:
        lc_path = root / split_name / "test_full_lightcurves.csv"
    else:
        lc_path = root / split_name / "train_full_lightcurves.csv"
    lc = pd.read_csv(lc_path)

    log = test_log if is_test else train_log

    # Keep only metadata rows for this split
    meta = log.loc[log["split"] == split_name].copy()

    # Merge target + static features onto each observation row
    merge_cols = ["object_id", "Z", "Z_err", "EBV", "SpecType", "split"]
    if not is_test and "target" in meta.columns:
        merge_cols.insert(1, "target")
    merge_cols = [col for col in merge_cols if col in meta.columns]
    merged = lc.merge(
        meta[merge_cols],
        on="object_id",
        how="left",
        validate="many_to_one"
    )
    return merged, meta

In [ ]:
FILTER_ORDER = ["u", "g", "r", "i", "z", "y"]
FILTER_TO_IDX = {f: i for i, f in enumerate(FILTER_ORDER)}

def encode_sequence(obj_df):
    obj_df = obj_df.sort_values("Time (MJD)").copy()

    # Robust numeric conversion + NaN handling to prevent NaN loss during training
    t = pd.to_numeric(obj_df["Time (MJD)"], errors="coerce").to_numpy(dtype=np.float32)
    flux = pd.to_numeric(obj_df["Flux"], errors="coerce").to_numpy(dtype=np.float32)
    flux_err = pd.to_numeric(obj_df["Flux_err"], errors="coerce").to_numpy(dtype=np.float32)

    t = np.nan_to_num(t, nan=0.0, posinf=0.0, neginf=0.0)
    flux = np.nan_to_num(flux, nan=0.0, posinf=0.0, neginf=0.0)
    flux_err = np.nan_to_num(flux_err, nan=0.0, posinf=0.0, neginf=0.0)

    dt = np.diff(t, prepend=t[0]).astype(np.float32)
    dt = np.clip(dt, a_min=0.0, a_max=1e4)

    # Stabilize dynamic ranges for RNN training
    flux = np.clip(flux, -1e5, 1e5)
    flux_err = np.clip(flux_err, 0.0, 1e5)

    filt_oh = np.zeros((len(obj_df), len(FILTER_ORDER)), dtype=np.float32)
    filt_idx = obj_df["Filter"].map(FILTER_TO_IDX).to_numpy()
    valid = ~pd.isna(filt_idx)
    filt_oh[np.arange(len(obj_df))[valid], filt_idx[valid].astype(int)] = 1.0

    x = np.column_stack([flux, flux_err, dt, filt_oh]).astype(np.float32)
    x = np.nan_to_num(x, nan=0.0, posinf=0.0, neginf=0.0)
    return torch.tensor(x, dtype=torch.float32)

class LightCurveSplitDataset(Dataset):
    def __init__(self, split_name, root_path, is_test=False):
        self.split_name = split_name
        self.root_path = root_path
        self.is_test = is_test

        self.merged, self.meta = load_split(split_name, is_test=is_test)

        self.meta = self.meta.set_index("object_id", drop=False)

        object_set = set(self.merged["object_id"].unique())
        self.object_ids = [oid for oid in self.meta["object_id"].tolist() if oid in object_set]
        self.grouped = dict(tuple(self.merged.groupby("object_id")))

    def __len__(self):
        return len(self.object_ids)

    def __getitem__(self, idx):
        oid = self.object_ids[idx]
        obj_df = self.grouped[oid]
        m = self.meta.loc[oid]

        seq = encode_sequence(obj_df)

        z = 0.0 if pd.isna(m.get("Z", np.nan)) else float(m["Z"])
        z_err = 0.0 if pd.isna(m.get("Z_err", np.nan)) else float(m["Z_err"])
        ebv = 0.0 if pd.isna(m.get("EBV", np.nan)) else float(m["EBV"])
        static_np = np.array([z, z_err, ebv], dtype=np.float32)
        static_np = np.nan_to_num(static_np, nan=0.0, posinf=0.0, neginf=0.0)
        static_np = np.clip(static_np, -1e3, 1e3)
        static = torch.tensor(static_np, dtype=torch.float32)

        sample = {
            "object_id": oid,
            "seq": seq,
            "static": static,
        }

        if not self.is_test and "target" in self.merged.columns:
            sample["y"] = torch.tensor(float(m["target"]), dtype=torch.float32)

        return sample

"""
Build one mini-batch from a list of dataset samples.
"""
def collate_pad(batch):
    seqs = [b["seq"] for b in batch]
    lengths = torch.tensor([s.size(0) for s in seqs], dtype=torch.long)
    seq_padded = pad_sequence(seqs, batch_first=True)

    static = torch.stack([b["static"] for b in batch], dim=0)
    y = torch.stack([b["y"] for b in batch], dim=0)
    obj_ids = [b["object_id"] for b in batch]

    return {
        "object_id": obj_ids,
        "seq": seq_padded,
        "lengths": lengths,
        "static": static,
        "y": y
    }

def collate_pad_test(batch):
    seqs = [b["seq"] for b in batch]
    lengths = torch.tensor([s.size(0) for s in seqs], dtype=torch.long)
    seq_padded = pad_sequence(seqs, batch_first=True)

    static = torch.stack([b["static"] for b in batch], dim=0)
    obj_ids = [b["object_id"] for b in batch]

    return {
        "object_id": obj_ids,
        "seq": seq_padded,
        "lengths": lengths,
        "static": static,
    }

# Example usage:
train_ds = LightCurveSplitDataset("split_01", root)
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, collate_fn=collate_pad)

batch = next(iter(train_loader))

In [7]:
class LSTMWithStatic(nn.Module):
    def __init__(self, seq_input_size=9, static_input_size=3, hidden_size=64, num_layers=1, dropout=0.1):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=seq_input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        self.static_net = nn.Sequential(
            nn.Linear(static_input_size, 16),
            nn.ReLU(),
            nn.Dropout(dropout),
        )
        self.head = nn.Sequential(
            nn.Linear(hidden_size + 16, 32),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(32, 1),
        )

    def forward(self, seq, lengths, static):
        packed = nn.utils.rnn.pack_padded_sequence(
            seq, lengths.cpu(), batch_first=True, enforce_sorted=False
        )
        _, (h_n, _) = self.lstm(packed)
        seq_repr = h_n[-1]
        static_repr = self.static_net(static)
        x = torch.cat([seq_repr, static_repr], dim=1)
        logits = self.head(x).squeeze(1)
        return logits

In [8]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = LSTMWithStatic(seq_input_size=9, static_input_size=3).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

split_names = sorted(train_log["split"].dropna().unique().tolist())
print("Training splits:", split_names)

# Build loaders once (faster and deterministic across epochs)
train_loaders = {}
all_targets = []
for split_name in split_names:
    ds = LightCurveSplitDataset(split_name, root)
    train_loaders[split_name] = DataLoader(ds, batch_size=64, shuffle=True, collate_fn=collate_pad)
    all_targets.append(ds.meta["target"].astype(float).values)

all_targets = np.concatenate(all_targets)
pos = float((all_targets == 1).sum())
neg = float((all_targets == 0).sum())
pos_weight = torch.tensor([neg / max(pos, 1.0)], dtype=torch.float32, device=device)
loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
print(f"Class balance -> pos: {int(pos)}, neg: {int(neg)}, pos_weight: {pos_weight.item():.2f}")

n_epochs = 100
for epoch in range(1, n_epochs + 1):
    model.train()
    running_loss = 0.0
    n_obs = 0

    for split_name in split_names:
        loader = train_loaders[split_name]
        for batch in loader:
            # print('batch:', batch)  # Debug print to check batch contents
            seq = batch["seq"].to(device)
            lengths = batch["lengths"]
            static = batch["static"].to(device)
            y = batch["y"].to(device)

            logits = model(seq, lengths, static)
            loss = loss_fn(logits, y)

            if torch.isnan(loss):
                print(f"NaN loss detected in {split_name}. Skipping this batch.")
                continue

            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            bs = y.size(0)
            running_loss += loss.item() * bs
            n_obs += bs

    epoch_loss = running_loss / max(n_obs, 1)

    if epoch == 1 or epoch % 2 == 0:
        model.eval()
        with torch.no_grad():
            logits_all = []
            y_all = []
            for split_name in split_names:
                for batch in train_loaders[split_name]:
                    seq = batch["seq"].to(device)
                    lengths = batch["lengths"]
                    static = batch["static"].to(device)
                    y = batch["y"].to(device)

                    logits = model(seq, lengths, static)
                    logits = torch.nan_to_num(logits, nan=0.0, posinf=0.0, neginf=0.0)
                    logits_all.append(logits.cpu())
                    y_all.append(y.cpu())

            logits_all = torch.cat(logits_all)
            y_all = torch.cat(y_all)
            probs = torch.sigmoid(logits_all)
            preds = (probs >= 0.5).float()

            acc = (preds == y_all).float().mean().item()
            precision = ((preds * y_all).sum() / (preds.sum() + 1e-8)).item()
            recall = ((preds * y_all).sum() / (y_all.sum() + 1e-8)).item()
            f1 = (2 * precision * recall) / (precision + recall + 1e-8)

        print(
            f"Epoch {epoch:02d} | loss={epoch_loss:.4f} | acc={acc:.4f} | "
            f"precision={precision:.4f} | recall={recall:.4f} | f1={f1:.4f}"
        )

Training splits: ['split_01', 'split_02', 'split_03', 'split_04', 'split_05', 'split_06', 'split_07', 'split_08', 'split_09', 'split_10', 'split_11', 'split_12', 'split_13', 'split_14', 'split_15', 'split_16', 'split_17', 'split_18', 'split_19', 'split_20']
Class balance -> pos: 148, neg: 2895, pos_weight: 19.56
Epoch 01 | loss=1.3199 | acc=0.9514 | precision=0.0000 | recall=0.0000 | f1=0.0000
Epoch 02 | loss=1.3008 | acc=0.8968 | precision=0.0677 | recall=0.0878 | f1=0.0765
Epoch 04 | loss=1.2672 | acc=0.8524 | precision=0.0808 | recall=0.1959 | f1=0.1144
Epoch 06 | loss=1.4092 | acc=0.8140 | precision=0.0664 | recall=0.2162 | f1=0.1016
Epoch 08 | loss=1.2860 | acc=0.8403 | precision=0.1194 | recall=0.3581 | f1=0.1791
Epoch 10 | loss=1.2357 | acc=0.8495 | precision=0.1327 | recall=0.3784 | f1=0.1965
Epoch 12 | loss=1.1879 | acc=0.9044 | precision=0.1674 | recall=0.2432 | f1=0.1983
Epoch 14 | loss=1.1514 | acc=0.8659 | precision=0.1486 | recall=0.3716 | f1=0.2124
Epoch 16 | loss=1.1529

In [9]:
# # Save trained model (and training state) to Google Drive
# ckpt_dir = root / "checkpoints"
# ckpt_dir.mkdir(parents=True, exist_ok=True)

# ckpt_path = ckpt_dir / "lstm_with_static_split01_20.pt"
# torch.save(
#     {
#         "model_state_dict": model.state_dict(),
#         "optimizer_state_dict": optimizer.state_dict(),
#         "epoch": epoch,
#         "loss": epoch_loss,
#         "acc": acc,
#         "precision": precision,
#         "recall": recall,
#         "f1": f1,
#         "pos_weight": pos_weight.detach().cpu(),
#         "split_names": split_names,
#     },
#     ckpt_path,
# )

# print(f"Saved checkpoint to: {ckpt_path}")

Saved checkpoint to: /content/drive/MyDrive/data/mallorn-astronomical-classification-challenge/checkpoints/lstm_with_static_split01_20.pt


In [20]:
model.eval()

test_split_names = sorted(test_log["split"].dropna().unique().tolist())
pred_rows = []
batch_size = 128

with torch.no_grad():
    for split_name in test_split_names:
        lc = pd.read_csv(root / split_name / "test_full_lightcurves.csv")
        meta = test_log.loc[test_log["split"] == split_name].copy().set_index("object_id", drop=False)
        grouped = dict(tuple(lc.groupby("object_id")))
        object_ids = [oid for oid in meta["object_id"].tolist() if oid in grouped]

        for start in range(0, len(object_ids), batch_size):
            batch_ids = object_ids[start:start + batch_size]
            seqs = []
            statics = []

            for object_id in batch_ids:
                obj_df = grouped[object_id]
                seqs.append(encode_sequence(obj_df))

                m = meta.loc[object_id]
                z = 0.0 if pd.isna(m.get("Z", np.nan)) else float(m["Z"])
                z_err = 0.0 if pd.isna(m.get("Z_err", np.nan)) else float(m["Z_err"])
                ebv = 0.0 if pd.isna(m.get("EBV", np.nan)) else float(m["EBV"])
                static_np = np.array([z, z_err, ebv], dtype=np.float32)
                static_np = np.nan_to_num(static_np, nan=0.0, posinf=0.0, neginf=0.0)
                statics.append(torch.tensor(np.clip(static_np, -1e3, 1e3), dtype=torch.float32))

            lengths = torch.tensor([seq.size(0) for seq in seqs], dtype=torch.long)
            seq_padded = pad_sequence(seqs, batch_first=True)
            static = torch.stack(statics, dim=0)

            logits = model(seq_padded.to(device), lengths, static.to(device))
            preds = (torch.sigmoid(logits).cpu().numpy() >= 0.5).astype(int)

            for object_id, pred in zip(batch_ids, preds):
                pred_rows.append({"object_id": object_id, "prediction": int(pred)})

pred_df = pd.DataFrame(pred_rows)
sample_submission = pd.read_csv(root / "sample_submission.csv")
pred_df = sample_submission[["object_id"]].merge(pred_df, on="object_id", how="left")
pred_df["prediction"] = pred_df["prediction"].fillna(0).astype(int)

output_path = Path.cwd() / "submission.csv"
pred_df.to_csv(output_path, index=False)
print(f"Saved predictions to: {output_path}")
print(pred_df.shape)
print(pred_df.to_csv(index=False).strip())

Saved predictions to: /content/submission.csv
(7135, 2)
object_id,prediction
Eluwaith_Mithrim_nothrim,0
Eru_heledir_archam,0
Gonhir_anann_fuin,0
Gwathuirim_haradrim_tegilbor,0
achas_minai_maen,0
adab_fae_gath,0
adel_draug_gaur,0
aderthad_cuil_galadhrim,1
aegas_laug_ithildin,0
aegas_mereth_law,0
agar_heron_salph,1
agarwaen_glae_ithildin,0
agor_achas_aiglos,0
agor_ang_adel,0
aiglos_Gonhir_lacha,0
alagos_mithren_harad,0
alph_ang_graw,0
amarth_cuil_cund,0
amdir_lasguil_neledh,0
andrann_acharn_ryn,0
andrann_rovalug_ithildin,0
angos_bass_aras,0
angos_elvellon_melethron,0
anwar_firion_gobennas,0
aphadrim_elu_tavor,0
aphadrim_ruith_cyron,0
aran_puig_cuil,0
aras_aeruil_muinthel,0
barad_araf_loss,0
bragol_manadh_heron,0
bragol_melethril_certh,1
braig_dam_rusc,1
bronwe_echad_mithril,0
cair_pethron_tirith,0
cair_ulug_min,0
corch_salph_mithren,0
cuil_firiel_minas,0
cund_per_parf,0
cund_roch_magol,0
curunir_luith_tirith,0
curunir_mirdan_tavor,0
curunir_nagol_idhren,0
cyron_baug_aes,0
dagnir_faradrim